# CHAPTER 03 · LangGraph로 워크플로 만들기

> **Building Workflows with LangGraph** — 상태(State) · 분기(Branching) · 메모리(Memory) · 체크포인트(Checkpoint)

---

## 이 챕터에서 배울 것 (Learning Objectives)

| # | 주제 | 핵심 내용 |
|---|------|-----------|
| 01 | **LangGraph 기초** | `State` · `Node` · `Edge`로 워크플로를 정의하고, 조건부 분기와 supersteps를 다룬다 |
| 02 | **출력 제어 & 에러 복구** | `OutputParser` · `Retry` · `Fallback`으로 비결정적 LLM 출력을 견고하게 처리한다 |
| 03 | **Prompt Engineering** | Zero/Few-shot, CoT, Self-consistency 등 고급 프롬프트 기법을 익힌다 |
| 04 | **Long Context** | Map-Reduce로 긴 입력을 분할 정복한다 |
| 05 | **메모리 & 체크포인트** | 트리밍·요약·체크포인트로 상태를 영속화하고 시간여행한다 |

## 챕터 로드맵

```
SECTION 01            SECTION 02           SECTION 03          SECTION 04        SECTION 05
LangGraph 기초   →   출력 & 에러     →    Prompt 엔지니어링  →  Long Context  →  메모리 & 체크포인트
State·Node·Edge      Parser·Retry·         Zero/Few·CoT·        Map-Reduce        History·MemorySaver
                     Fallback              Self-consistency      ·Send             ·Time Travel
```


## 0. 환경 설정 (Setup)

### 실행 표기 규칙

이 노트북의 코드 셀은 두 종류로 구분됩니다.

- **✅ 실행 가능 (offline)** — LLM API 키 없이 바로 돌아갑니다. LangGraph 구조, reducer, 파서, 체크포인트 등 핵심 메커니즘은 가짜(fake) 모델이나 순수 그래프로 검증할 수 있습니다.
- **🔑 API 키 필요 (illustrative)** — 실제 LLM 호출이 들어가는 셀입니다. 패턴 이해용이며, 키를 설정하면 그대로 실행됩니다.

먼저 필요한 패키지를 설치합니다. (이미 설치돼 있다면 건너뛰어도 됩니다.)

In [ ]:
# 패키지 설치 (필요 시 주석 해제)
%pip install -qU langgraph langchain langchain-core langchain-google-genai grandalf langchain-classic

### API 키 설정

원서는 `config.py`의 `set_environment()`로 키를 한 번에 등록하는 방식을 권장합니다.
아래처럼 직접 환경변수를 넣어도 됩니다. **키가 없어도 ✅ 표시된 셀은 모두 동작합니다.**

In [ ]:
# setting the environment variables, the keys
from dotenv import load_dotenv
import sys
import os

load_dotenv()

---
# SECTION 01 · LangGraph 기초
**State · Node · Edge · Superstep**

LangChain의 일반 체인(LCEL)이 **선형 파이프라인(DAG)**이라면, LangGraph는 **상태를 공유하는 그래프**로 워크플로를 표현합니다.

## 1.1 왜 LangGraph인가?

또 다른 오케스트레이션 프레임워크가 필요한 이유는 네 가지입니다.

| 특징 | 설명 |
|------|------|
| 🔁 **사이클 지원** | 대부분의 프레임워크는 DAG만 지원. LangGraph는 cycle을 허용해 **재시도·반복**이 가능 |
| 📡 **스트리밍 기본 제공** | 노드별 출력을 즉시 스트리밍해 응답 지연을 체감상 0으로 |
| 🧱 **Gen AI 빌딩 블록** | human-in-the-loop 등 GenAI 워크플로 전용 컴포넌트 내장 |
| 🎛 **Low-level API** | 필요하면 실행 흐름을 매우 세밀하게 통제 가능 |

## 1.2 그래프 해부도 — State · Nodes · Edges

LangGraph 워크플로는 세 가지 핵심 구성요소로 이루어집니다.

- **STATE (상태)** — 워크플로 전체가 공유하는 컨텍스트. Python `dict`로 생각하면 쉽습니다. `TypedDict` · `Pydantic` · `dataclass`로 정의.
- **NODES (노드)** — 상태를 입력받아 *업데이트할 키-값*을 반환하는 Python 함수. 각 노드는 하나의 "행위(action)".
- **EDGES (엣지)** — 노드 사이의 흐름. 직접 엣지 · 조건부 엣지 · `START` · `END`. 동시(parallel) 실행도 가능.

## 1.3 State 관리 — TypedDict 스키마

상태는 보통 `TypedDict`로 스키마를 정의합니다. 노드는 상태의 **불변 복사본**을 받아 *바꿀 키만* dict로 반환하고, 병합은 LangGraph가 책임집니다.

- **Immutable Copy** — 노드는 참조가 아닌 복사본을 받아 부수효과를 차단
- **Explicit Updates** — "바꿀 키만" 반환 → LangGraph가 머지
- **Traceable** — 모든 변경이 명시적·추적 가능 → 디버깅·시간여행에 유리

> ✅ 아래 셀은 LLM 없이 실행됩니다. 노드 함수와 상태 스키마만 정의합니다.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class SupportTicketState(TypedDict):
    customer_message: str   # 입력: 고객 문의 메시지
    is_urgent: bool      # 분석 결과: 긴급 여부
    response: str        # 산출물: 답변 초안


def triage_ticket(state):
    print("...고객 지원 티켓 분석 중...")
    # 길이가 100자를 넘으면 '긴급'으로 간주하는 가짜 규칙
    return {"is_urgent": len(state["customer_message"]) > 100}


def draft_response(state):
    print("...답변 초안 생성 중...")
    return {"response": "some_draft_response"}


print("상태 스키마와 노드 함수 정의 완료 ✅")

## 1.4 그래프 빌드 4단계

`StateGraph → add_node → add_edge → compile`

1. `StateGraph(스키마)` 로 빌더 생성
2. `add_node(이름, 함수)` 로 노드 등록
3. `add_edge(시작, 끝)` 로 흐름 연결 (`START` / `END` 사용)
4. `compile()` → 실행 가능한 `Runnable` 반환

> ✅ 컴파일된 그래프는 LangChain의 `Runnable`이므로 `invoke` / `stream` / `batch`를 그대로 지원합니다.

In [ ]:
builder = StateGraph(SupportTicketState)
builder.add_node("triage_ticket", triage_ticket)
builder.add_node("draft_response", draft_response)

builder.add_edge(START, "triage_ticket")
builder.add_edge("triage_ticket", "draft_response")
builder.add_edge("draft_response", END)

graph = builder.compile()

# 컴파일 결과는 Runnable
from langchain_core.runnables import Runnable
print("graph 는 Runnable 인가? ->", isinstance(graph, Runnable))

In [ ]:
# ASCII로 그래프 구조 출력 (Mermaid PNG 대신 offline 친화적)
print(graph.get_graph().draw_ascii())

In [ ]:
# 실행: 최종 상태(dict)를 반환
res = graph.invoke({"customer_message": "fake_jd"})
print(res)

> 💡 그래프를 이미지로 보고 싶다면 Jupyter에서 아래를 실행하세요 (인터넷/렌더러 필요):
> ```python
> from IPython.display import Image, display
> display(Image(graph.get_graph().draw_mermaid_png()))
> ```

## 1.5 Supersteps & 병렬 실행

LangGraph는 Google **Pregel** 시스템에서 영감을 받은 실행 모델을 씁니다.

- **Superstep** = 한 번의 반복 단위. 한 단계에서 여러 노드를 동시에 실행하고, 각 노드의 상태 업데이트를 한꺼번에 머지합니다.
- `recursion_limit`으로 무한 루프를 방지합니다.

```
Step 1:  START → analyze
Step 2:  analyze → [nodeA · nodeB]   # 병렬
Step 3:  merge state → reduce
Step 4:  → END
```

## 1.6 조건부 엣지 (Conditional Edges)

상태에 따라 실행 경로를 **동적으로** 결정합니다. 함수가 현재 상태를 받아 *다음 노드 이름(또는 `END`)*을 반환합니다.

> **TIP** — 반환 타입에 `Literal` 힌트를 주면 LangGraph가 가능한 목적지 노드를 자동 추론합니다. 없으면 모든 노드와 연결돼 시각화가 혼란스러워집니다.

> ✅ offline 실행 가능.

In [ ]:
from typing import Literal

builder = StateGraph(SupportTicketState)
builder.add_node("triage_ticket", triage_ticket)
builder.add_node("draft_response", draft_response)


def is_urgent_condition(state: SupportTicketState) -> Literal["draft_response", END]:
    # 긴급하면 답변 생성 노드로, 아니면 종료
    if state.get("is_urgent"):
        return "draft_response"
    return END


builder.add_edge(START, "triage_ticket")
builder.add_conditional_edges("triage_ticket", is_urgent_condition)
builder.add_edge("draft_response", END)

graph = builder.compile()
print(graph.get_graph().draw_ascii())

In [ ]:
# 짧은 입력 → is_urgent=False → 곧바로 END (generate 건너뜀)
print("짧은 입력:", graph.invoke({"customer_message": "short"}))
# 긴 입력 → is_urgent=True → draft_response 실행
print("긴 입력 :", graph.invoke({"customer_message": "x" * 200}))

## 1.7 Reducers — 상태 머지 전략

키 단위로 "값 시퀀스를 어떻게 합칠 것인가"를 정의합니다.

- **기본 (replace)** — 노드가 반환한 값으로 덮어쓰기. reducer를 지정하지 않으면 적용.
- **`operator.add`** — 리스트를 이어붙여 누적. `Annotated[list, add]`로 선언.
- **커스텀 reducer** — 직접 병합 함수를 작성.

먼저 reducer 없이 `actions` 리스트를 누적하려 하면 **마지막 값으로 덮어써지는** 것을 봅니다.

In [ ]:
# reducer 없음 → actions 가 매번 덮어써짐
class StateNoReducer(TypedDict):
    customer_message: str
    is_urgent: bool
    actions: list[str]


def n1(state):
    return {"is_urgent": True, "actions": ["action1"]}


def n2(state):
    return {"actions": ["action2"]}


b = StateGraph(StateNoReducer)
b.add_node("n1", n1); b.add_node("n2", n2)
b.add_edge(START, "n1"); b.add_edge("n1", "n2"); b.add_edge("n2", END)

print("reducer 없음:", b.compile().invoke({"customer_message": "x", "actions": []}))

In [ ]:
# operator.add reducer → actions 가 누적됨
from typing import Annotated
from operator import add


class StateWithAdd(TypedDict):
    customer_message: str
    is_urgent: bool
    actions: Annotated[list[str], add]   # ← reducer 지정


b = StateGraph(StateWithAdd)
b.add_node("n1", n1); b.add_node("n2", n2)
b.add_edge(START, "n1"); b.add_edge("n1", "n2"); b.add_edge("n2", END)
print("operator.add :", b.compile().invoke({"customer_message": "x", "actions": []}))

In [ ]:
# 커스텀 reducer: 문자열이면 리스트로 감싸고, 리스트면 이어붙임
from typing import Optional, Union


def my_reducer(left: list[str], right: Optional[Union[str, list[str]]]) -> list[str]:
    if right:
        return left + [right] if isinstance(right, str) else left + right
    return left


class StateCustom(TypedDict):
    customer_message: str
    actions: Annotated[list[str], my_reducer]


def c1(state):
    return {"actions": "action1"}          # 문자열 하나


def c2(state):
    return {"actions": ["action2", "action3"]}  # 리스트


b = StateGraph(StateCustom)
b.add_node("c1", c1); b.add_node("c2", c2)
b.add_edge(START, "c1"); b.add_edge("c1", "c2"); b.add_edge("c2", END)
print("custom reducer:", b.compile().invoke({"customer_message": "x", "actions": []}))

## 1.8 Configurable Graphs — RunnableConfig

`RunnableConfig`로 **사용자 입력(state)**과 **실행 파라미터(config)**를 분리합니다. 동일 그래프를 다른 LLM·콜백으로 재사용할 수 있습니다.

- `configurable` 키 — 임의의 사용자 파라미터 dict 전달
- `recursion_limit` — superstep 최대 수를 제어해 폭주 방지

> ✅ 아래는 LLM 호출 없이 config 분기만 보여주므로 offline 실행됩니다.

In [ ]:
from langchain_core.runnables.config import RunnableConfig


class CfgState(TypedDict):
    customer_message: str
    response: str


def generate_response_with_config(state: CfgState, config: RunnableConfig):
    cfg = config.get("configurable", {})
    provider = cfg.get("model_provider", "Google")
    model = cfg.get("model_name", "gemini-2.0-flash")
    print(f"...{provider} / {model} 로 답변 생성...")
    return {"response": f"reply via {provider}:{model}"}


b = StateGraph(CfgState)
b.add_node("generate", generate_response_with_config)
b.add_edge(START, "generate"); b.add_edge("generate", END)
g = b.compile()

print(g.invoke({"customer_message": "jd"}))  # 기본값
print(g.invoke({"customer_message": "jd"},
               config={"configurable": {"model_provider": "OpenAI", "model_name": "gpt-4o"}}))

---
# SECTION 02 · 출력 제어 & 에러 핸들링
**Parsers · Retries · Fallbacks**

LLM은 비결정적입니다. 워크플로의 다음 단계가 안전하게 소비할 수 있도록 출력을 **구조화**하고, 실패에 대비한 **재시도·폴백**을 갖춰야 합니다.

## 2.1 Output Parsing — 자연어 → 구조화 데이터

자유 텍스트는 후속 단계가 신뢰하기 어렵습니다. `OutputParser`로 프로그램이 처리 가능한 형태로 변환합니다.

지원 파서: `Enum` · `List` · `CSV` · `JSON` · `XML` · `Pydantic` · `DataFrame` (+ 사용자 정의)

> ✅ 파서 자체는 LLM 없이 검증할 수 있습니다.

In [ ]:
import os
print([f for f in os.listdir('.') if f.startswith('langchain')])

In [ ]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser
from langchain_core.messages import HumanMessage


class IsUrgentEnum(Enum):
    YES = "YES"
    NO = "NO"


parser = EnumOutputParser(enum=IsUrgentEnum)

# 파서는 공백/개행을 정리하고 메시지 객체도 처리합니다
assert parser.invoke("NO") == IsUrgentEnum.NO
assert parser.invoke("YES\n") == IsUrgentEnum.YES
assert parser.invoke(" YES \n") == IsUrgentEnum.YES
assert parser.invoke(HumanMessage(content=" YES \n")) == IsUrgentEnum.YES
print("EnumOutputParser 검증 통과 ✅")

LLM과 파이프(`|`)로 합성하면 `llm | parser` 형태의 체인이 됩니다.

> 🔑 아래는 실제 LLM 호출이 필요합니다 (illustrative).

In [ ]:
# 🔑 API 키 필요 — 패턴 예시
prompt_template_enum = (
    "Given a customer support ticket, decide whether it is URGENT and needs immediate attention."
    "\nTICKET:\n{customer_message}\n\nAnswer only YES or NO."
)


from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = llm | parser
result = chain.invoke(prompt_template_enum.format(customer_message="The app keeps crashing on login!"))
print(result)


## 2.2 에러 핸들링

API 호출 실패, 예상 외 출력, 외부 서비스 장애 — 어느 단계든 실패할 수 있습니다. 노드 안에서 `try/except`로 **로깅 + 기본값 반환**하면 워크플로를 계속 진행할 수 있습니다. (예외를 텍스트로 변환해 LLM에 컨텍스트로 줄 수도 있습니다.)

In [ ]:
import logging
logger = logging.getLogger(__name__)


def triage_ticket_safe(state):
    try:
        analyze_chain = llm | parser   # 실패 가능 지점
        prompt = prompt_template_enum.format(customer_message=state["customer_message"])
        result = analyze_chain.invoke(prompt)
        return {"is_urgent": result}
    except Exception as e:
        logger.error(f"Exception {e} 발생 — 기본값으로 대체")
        return {"is_urgent": False}   # 안전한 기본값

print("에러를 흡수하고 기본값을 반환하는 노드 정의 완료 ✅")

## 2.3 재시도 (Retries) — 3가지 접근법

실패를 자동으로 다시 시도하는 세 가지 층위가 있습니다.

| # | 방법 | 특징 |
|---|------|------|
| 01 | `Runnable.with_retry` | 임의의 Runnable에 재시도 부착. 지수 백오프 |
| 02 | 노드별 `RetryPolicy` | 특정 노드에만 정책 부여. 에러 종류·횟수 통제 |
| 03 | `RetryWithErrorOutputParser` | LLM에게 에러를 알려 응답 재생성 — **의미 기반 복구** |

실험을 위해, **홀수 번째 호출마다 `ValueError`를 던지는** 가짜 LLM을 만듭니다.

> ✅ 가짜 모델이라 offline 실행됩니다.

In [ ]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class MessagesIterator:
    # 홀수 호출 → ValueError, 짝수 호출 → 'YES' 반환
    def __init__(self):
        self._count = 0

    def __iter__(self):
        return self

    def __next__(self):
        self._count += 1
        if self._count % 2 == 1:
            raise ValueError("Something went wrong")
        return AIMessage(content="YES")


fake_llm = GenericFakeChatModel(messages=MessagesIterator())
print("fake_llm 준비 완료 (홀수 호출마다 에러) ✅")

In [ ]:
# 01. with_retry — 첫 호출 실패(ValueError) 후 자동 재시도해서 성공
analyze_chain_fake_retries = (fake_llm | parser).with_retry(
    retry_if_exception_type=(ValueError,),
    wait_exponential_jitter=True,
    stop_after_attempt=2,
)
print("with_retry 결과:", analyze_chain_fake_retries.invoke("test"))

In [ ]:
# 02. 노드별 RetryPolicy — analyze 노드에만 재시도 정책 부여
from langgraph.types import RetryPolicy
from typing import TypedDict, Annotated
from operator import add

class SupportTicketState2(TypedDict):
    customer_message: str
    is_urgent: bool
    response: str
    actions: Annotated[list[str], add]


def triage_with_fake(state):
    result = (fake_llm | parser).invoke("test")   # 홀수 호출 시 ValueError
    return {"is_urgent": result == IsUrgentEnum.YES}


def gen_response(state):
    return {"response": "draft_reply", "actions": ["generated"]}


b = StateGraph(SupportTicketState2)
b.add_node("analyze", triage_with_fake,
           retry=RetryPolicy(retry_on=ValueError, max_attempts=2))   # ← 노드 재시도
b.add_node("generate", gen_response)
b.add_edge(START, "analyze")
b.add_edge("analyze", "generate")
b.add_edge("generate", END)
graph = b.compile()
print("RetryPolicy 그래프 결과:", graph.invoke({"customer_message": "jd", "actions": []}))

### 03. 의미적 복구 — RetryWithErrorOutputParser

파싱 실패 시, **에러 메시지를 LLM에게 알려** 스키마에 맞게 응답을 다시 생성하도록 합니다. 단순 재시도와 달리 *내용을 고쳐서* 복구합니다.

> 🔑 LLM 호출 필요 (illustrative).

In [ ]:
# 🔑 API 키 필요 — 의미적 복구 패턴
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import RetryWithErrorOutputParser
from pydantic import BaseModel, Field

class SearchAction(BaseModel):
    query: str = Field(description="A query to search for if a search action is taken")


pyd_parser = PydanticOutputParser(pydantic_object=SearchAction)
broken = "{'action': 'what is the weather likein Munich tomorrow}"   # 스키마 위반

# 먼저 일반 파싱은 실패함을 확인 (offline 가능)
try:
    pyd_parser.parse(broken)
except Exception as e:
    print("일반 파싱 실패:", type(e).__name__)


fix_parser = RetryWithErrorOutputParser.from_llm(llm=llm, parser=pyd_parser)
retry_prompt = PromptTemplate.from_template(
    "Your previous response doesn't follow the required schema and fails parsing. "
    "Fix the response so that it follows the expected schema. "
    "Do not change the nature of response, only adjust the schema.\n\n"
    "EXPECTED SCHEMA:{schema}\n\n"
)
fixed = fix_parser.parse_with_prompt(
    completion=broken,
    prompt_value=retry_prompt.format_prompt(schema=pyd_parser.get_format_instructions()),
)
print("복구 결과:", fixed)


## 2.4 Fallbacks — 대체 체인

메인 Runnable이 예외를 던지면, **동일한 입력**으로 백업 Runnable을 자동 실행합니다. 프로바이더 다중화(OpenAI 다운 → Anthropic 전환)에 유용합니다.

> ✅ fake_llm을 다시 활용해 offline 실행. (이 시점 `fake_llm`은 다음 호출에서 `ValueError`를 던지는 상태일 수 있어, 폴백이 트리거됩니다.)

In [ ]:
from langchain_core.runnables import RunnableLambda

# 메인 체인: fake_llm 호출이 실패하면 예외 → 폴백으로
chain_fallback = RunnableLambda(lambda _: "running fallback ✅")
main_chain = fake_llm | RunnableLambda(lambda msg: "running main chain ✅")
chain_with_fb = main_chain.with_fallbacks([chain_fallback])

# 두 번 호출: 홀짝에 따라 main / fallback 이 번갈아 나옴
print(chain_with_fb.invoke("test"))
print(chain_with_fb.invoke("test"))

---
# SECTION 03 · 프롬프트 엔지니어링
**Templates · Zero/Few-shot · CoT · Self-consistency**

## 3.1 Prompt Templates 복습 + 확장

- **`PromptTemplate`** — 문자열 템플릿 + 변수 치환
- **`ChatPromptTemplate`** — `(role, template)` 튜플로 메시지 시퀀스 구성
- **`MessagesPlaceholder`** — 런타임 메시지 리스트 삽입 (대화 히스토리·도구 결과)
- **`.partial(...)`** — 일부 변수만 미리 채움 (현재 시각 같은 동적 값도 가능)

> ✅ 템플릿 구성/치환은 LLM 없이 검증됩니다.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

prompt_template = (
    "Given a customer support ticket, decide whether it is URGENT and needs immediate attention."
    "\nTICKET:\n{customer_message}\n"
)

# 1) PromptTemplate
pt = PromptTemplate.from_template(prompt_template)
print("PromptTemplate:", pt.invoke({"customer_message": "fake_jd"}).text[:60], "...")

# 2) ChatPromptTemplate + MessagesPlaceholder
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),            # == MessagesPlaceholder("history")
    ("human", prompt_template),
])
msgs = chat_prompt.invoke({
    "customer_message": "fake",
    "history": [("human", "hi!"), ("ai", "hi!")],
}).messages
print("메시지 개수 (system+history2+human):", len(msgs))   # → 4

In [ ]:
# 3) .partial() — 일부 변수 미리 채우기
tmpl = PromptTemplate.from_template("a: {a} b: {b}")
partial = tmpl.partial(a="A")                 # a 만 고정
print(partial.invoke({"b": "B"}).text)         # 'a: A b: B'

# 4) 템플릿 결합(+)
combined = PromptTemplate.from_template("a: {a}") + " " + PromptTemplate.from_template("b: {b}")
print(combined.invoke({"a": "A", "b": "B"}).text)

## 3.2 Zero-shot vs Few-shot

| ZERO-SHOT | FEW-SHOT |
|-----------|----------|
| 예시 없이 지시만 | 입력-출력 예시 몇 개 제시 |
| 역할(role) 부여: "당신은 …입니다" | 짧은 입력 작업(분류·추출)에 효과적 |
| 추가 지시: 창의적/사실적/간결 | RAG처럼 긴 입력은 예시 비용 큼 |
| Gemini 가이드: persona·task·context·format | Dynamic Few-shot: 입력과 유사한 예시 선택 |
| 프로바이더별 권장 형식 확인 필수 | `FewShotPromptTemplate` + `ExampleSelector` |

## 3.3 Chain-of-Thought (CoT)

최종 답 토큰을 바로 만들지 말고 **중간 추론 과정을 먼저 생성**하게 유도합니다. *"Let's think step by step"* 한 줄만으로도 많은 작업에서 성능이 향상됩니다. (o3-mini, gemini-flash-thinking 등 reasoning 모델은 CoT를 학습 데이터로 내재화)

> 🔑 LLM + LangChain Hub 필요 (illustrative).

In [ ]:
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser

client = Client()
math_cot_prompt = client.pull_prompt("arietem/math_cot", dangerously_pull_public_prompt=True)

cot_chain = math_cot_prompt | llm | StrOutputParser()
print(cot_chain.invoke("Solve equation 2*x+5=15"))

## 3.4 Self-Consistency

Temperature를 높여 **여러 번 샘플링**한 뒤 **다수결**을 취합니다. 단일 샘플은 우연히 틀릴 수 있지만, 분포에서 최빈값을 고르면 노이즈가 평균화됩니다.

잘 맞는 작업: 분류, 엔티티 추출, 짧은 정답 수학 — **출력 차원이 낮은** 작업.

> 🔑 LLM 필요 (illustrative). offline에서는 다수결 로직만 시연합니다.

In [ ]:
import re
from collections import Counter

# offline 데모 (그대로 둬도 됨)
fake_generations = ["x = 24", "x = 24", "x = 23", "x = 24", "x = 25"]
print("최빈값:", Counter(fake_generations).most_common(1)[0][0])

# 1) temperature는 LLM에 바인딩 (다양성 확보용; 2.0은 과해서 1.0 권장)
hot_llm = llm.bind(temperature=1.0)
final_chain = math_cot_prompt | hot_llm | StrOutputParser()

# 2) 답에서 핵심 값만 뽑아 정규화 (CoT 출력은 문장이 길어 그대로 집계하면 다 달라짐)
def extract_answer(text: str) -> str:
    m = re.findall(r"x\s*=\s*[-+]?\d+(?:\.\d+)?", text.replace(" ", " "))
    return m[-1].replace(" ", "") if m else text.strip()

# 3) 20개를 batch로 한 번에 (순차 루프보다 빠르고 간결)
question = {"question": "Solve the equation 2*x**2 - 96*x + 1152 = 0"}
raw = final_chain.batch([question] * 20)
answers = [extract_answer(r) for r in raw]

print("LLM 다수결:", Counter(answers).most_common(1)[0][0])
print("분포:", Counter(answers))

---
# SECTION 04 · Long Context
**Working with Short Context Windows — Map-Reduce**

## 4.1 긴 컨텍스트 처리 전략

| STUFF (통째로) | MAP-REDUCE (분할 정복) |
|----------------|------------------------|
| 모든 입력을 하나의 프롬프트로 결합 | **Map**: 컨텍스트를 청크로 분할 → 병렬 작업 |
| 컨텍스트 윈도우에 맞을 때만 가능 | **Reduce**: 각 청크 결과를 하나로 결합 |
| 구현이 가장 단순 | 오픈모델·엣지 등 짧은 컨텍스트에 필수 |
| 1~2M 토큰 모델 시대에 다시 부상 | 비용·레이턴시·신뢰도 균형 |

## 4.2 LangGraph로 Map-Reduce — Send

`Send(node, state)`로 한 노드를 **다른 상태로 동적으로 여러 번** 스케줄링합니다. 비디오를 청크로 나눠 병렬 요약하는 예시입니다.

- `Send(node, state)` — 동적 병렬 스케줄링
- `Annotated[list, operator.add]` — 병렬 결과를 리스트로 누적 (순서 보장)
- `max_concurrency` — 동시 실행 개수 제한 (API 레이트 리밋 대응)

> 🔑 비디오 + Gemini 필요 (illustrative). 구조와 흐름을 정리합니다.

In [ ]:
# 🔑 API 키 + 비디오 URI 필요 — Map-Reduce 그래프 구조
import operator
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class AgentState(TypedDict):
    video_uri: str
    chunks: int
    interval_secs: int
    summaries: Annotated[list, operator.add]   # ← 병렬 결과 누적
    final_summary: str


class _ChunkState(TypedDict):
    video_uri: str
    start_offset: int
    interval_secs: int


def _map_summaries(state: AgentState):
    # 각 청크에 대해 summarize 노드를 Send 로 동적 스케줄링
    payloads = [
        {"video_uri": state["video_uri"],
         "interval_secs": state["interval_secs"],
         "start_offset": i}
        for i in range(state["chunks"])
    ]
    return [Send("summarize_video_chunk", p) for p in payloads]


print("Map-Reduce 그래프 골격 정의 완료 (요약/병합 노드는 LLM 필요) ✅")
print("흐름: START -(_map_summaries)-> [summarize_video_chunk]*N -> generate_final_summary -> END")

핵심은 **조건부 엣지에 `Send` 리스트를 반환**해 같은 노드를 여러 입력으로 fan-out 시키고, `operator.add` reducer로 결과를 모으는 것입니다:

```python
graph.add_conditional_edges(START, _map_summaries, ["summarize_video_chunk"])
graph.add_edge("summarize_video_chunk", "generate_final_summary")
graph.add_edge("generate_final_summary", END)
app = graph.compile()

result = await app.ainvoke(
    {"video_uri": video_uri, "chunks": 5, "interval_secs": 600},
    {"max_concurrency": 3},   # 동시 실행 제한
)
```

---
# SECTION 05 · 메모리 & 체크포인트
**Chat History · MemorySaver · Time Travel**

## 5.1 Chat History를 줄이는 5가지 방법

긴 대화를 컨텍스트에 맞추거나 비용을 절감하는 전략들입니다.

1. **길이 기반 폐기** — 최근 N개 메시지/토큰만 유지. `trim_messages`로 system 메시지 보존 가능
2. **이전 대화 요약** — 매 턴마다 이전 대화를 한 메시지로 요약 (LangGraph 노드로 구현 권장)
3. **트리밍 + 요약 결합** — 오래된 건 요약, 최근은 그대로
4. **긴 메시지 요약** — RAG 등 입력이 클 때 각 메시지 자체를 압축
5. **커스텀 토크나이저** — `trim_messages`에 자체 토크나이저 주입 (프로바이더별 정확한 카운트)

## 5.2 히스토리 영속화 & 트리밍

프로덕션 서버는 수평 확장을 위해 **무상태(stateless)**여야 하며, 히스토리는 외부 저장소에 영속합니다. `RunnableWithMessageHistory`가 세션별 히스토리를 관리하고, `trim_messages`가 길이를 제어합니다.

- `BaseChatMessageHistory` — Postgres·Redis·DynamoDB 등 다양한 DB 통합이 상속
- `session_id` 보안 — 순차 ID 대신 **UUID** 사용, 권한 검증 필수

> ✅ `FakeListChatModel`을 써서 offline 실행됩니다.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.language_models import FakeListChatModel
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import trim_messages, HumanMessage


class PrintOutputCallback(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print(f"모델에 전달된 입력 메시지 수: {len(messages[0])}")


sessions = {}
fake = FakeListChatModel(responses=["ai1", "ai2", "ai3"])


def get_session_history(session_id: str):
    if session_id not in sessions:
        sessions[session_id] = InMemoryChatMessageHistory()
    return sessions[session_id]


# 최근 1토큰(=메시지 1개)만 남기는 트리머
trimmer = trim_messages(
    max_tokens=1, strategy="last", token_counter=len,
    include_system=True, start_on="human",
)

raw_chain = trimmer | fake
chain = RunnableWithMessageHistory(raw_chain, get_session_history)

config = {"callbacks": [PrintOutputCallback()], "configurable": {"session_id": "1"}}
chain.invoke([HumanMessage("Hi!")], config=config)
print("히스토리 길이:", len(sessions["1"].messages))
chain.invoke([HumanMessage("How are you?")], config=config)
print("히스토리 길이:", len(sessions["1"].messages))

## 5.3 LangGraph Checkpoints

`checkpointer`를 붙이면 그래프 실행의 **스냅샷**이 저장돼 어디서든 멈추고 재개할 수 있습니다.

| 효과 | 설명 |
|------|------|
| 🐛 Deep Debug | 어느 시점에서든 다시 재생 |
| 🕰 Time Travel | 다른 경로로 분기 실험 |
| 👤 HITL | 사람 개입 후 이어서 진행 |
| 🛡 Fault Tolerance | 장애 후 회복·영속성 |

- `thread_id` — 독립된 실행 흐름(대화) 격리
- `checkpoint_id` — 특정 시점으로 시간여행

> ✅ `MemorySaver`는 로컬 메모리라 offline 실행됩니다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage
from langgraph.graph import MessagesState


def test_node(state: MessagesState):
    # 마지막(입력) 메시지를 제외한 누적 히스토리 길이 출력
    print(f"이전 히스토리 길이 = {len(state['messages'][:-1])}")
    return {"messages": [AIMessage(content="Hello!")]}


builder = StateGraph(MessagesState)
builder.add_node("test_node", test_node)
builder.add_edge(START, "test_node")
builder.add_edge("test_node", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)   # ← 체크포인터 연결
print("체크포인터가 붙은 그래프 컴파일 완료 ✅")

In [ ]:
# thread-a 는 상태가 누적되고, thread-b 는 완전히 분리됨
graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-a"}})
graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-b"}})
graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-a"}})

## 5.4 Time Travel & 멀티스레드 실행

`thread_id`로 격리하고, `checkpoint_id`로 과거 시점에서 분기합니다.

저장소 옵션: `MemorySaver`(개발용) · `SqliteSaver`(파일 기반) · `PostgresSaver`(프로덕션 권장).

> ✅ offline 실행 — thread-a의 체크포인트 목록을 조회하고 과거 시점으로 되돌아갑니다.

In [ ]:
# thread-a 의 체크포인트 히스토리 조회
checkpoints = list(memory.list(config={"configurable": {"thread_id": "thread-a"}}))
print("thread-a 체크포인트 수:", len(checkpoints))
for cp in checkpoints:
    print("  checkpoint_id:", cp.config["configurable"]["checkpoint_id"])

In [ ]:
# 과거 체크포인트로 시간여행 — 그 시점의 히스토리에서 이어서 실행
if len(checkpoints) >= 3:
    old_id = checkpoints[-3].config["configurable"]["checkpoint_id"]
    print(f"checkpoint {old_id[:8]}... 로 되돌아가 재실행:")
    graph.invoke(
        {"messages": [HumanMessage(content="test")]},
        config={"configurable": {"thread_id": "thread-a", "checkpoint_id": old_id}},
    )

---
# CHAPTER SUMMARY · 체인을 넘어 그래프로

| # | 영역 | 핵심 정리 |
|---|------|-----------|
| 01 | **LangGraph 기초** | `State`·`Node`·`Edge`로 워크플로 정의. 조건부 엣지로 분기, supersteps로 병렬 실행. Reducers로 상태 머지 전략 제어 |
| 02 | **출력 제어 & 에러 복구** | `OutputParser`로 구조화, `with_retry`·`RetryPolicy`·`RetryWithErrorOutputParser`·`with_fallbacks`로 견고함 확보 |
| 03 | **Prompt Engineering** | Zero/Few-shot · `MessagesPlaceholder` · CoT(step-by-step) · Self-consistency로 신뢰도 ↑ |
| 04 | **Long Context** | `Send` 기반 Map-Reduce로 긴 입력을 분할 정복 |
| 05 | **메모리 & 체크포인트** | `RunnableWithMessageHistory`·`MemorySaver`/Postgres로 세션 영속과 시간여행 |

## 복습 문제 (Review Questions)

1. LangGraph 워크플로가 LangChain 일반 체인과 다른 점은?
2. LangGraph에서 "state"의 정의와 주요 역할은?
3. supersteps와 병렬 실행의 관계를 설명하라.
4. 조건부 엣지가 시퀀셜 체인 대비 어떤 장점을 제공하나?
5. Reducer의 역할과 `add` / `add_messages`의 차이는?
6. LangGraph Checkpoint의 4가지 활용 사례는?